# Birliktelik Kuralları (Association Rule Mining)

**Hazırlayan:** Dr. Buket TOPTAŞ

---
Bu notebook, veri madenciliği dersinin önemli konularından biri olan **birliktelik kuralları** kavramını ele almaktadır.

| # | Konu |
|---|------|
| 1 | Birliktelik Kuralı Nedir? |
| 2 | Temel Kavramlar (Support, Confidence, Lift) |
| 3 | Apriori Algoritması |
| 4 | ECLAT Algoritması |
| 5 | FP-Growth Algoritması |
| 6 | Algoritmaların Karşılaştırılması |
| 7 | Gerçek Dünya Uygulaması |

---
## 1. Birliktelik Kuralı Nedir?

**Birliktelik kuralları**, büyük veri setlerindeki değişkenler arasındaki ilginç ilişkileri keşfetmek için kullanılan bir veri madenciliği tekniğidir.

En klasik örnek **Market Sepeti Analizi**'dir:
> *"Ekmek alan bir müşteri, %80 ihtimalle süt de alır."*

Bu tür kurallar şu şekilde gösterilir:

$$\{Ekmek\} \Rightarrow \{Süt\}$$

### Kullanım Alanları
- 🛒 Market sepeti analizi
- 🎬 Film/müzik öneri sistemleri
- 💊 Tıpta ilaç etkileşimleri
- 🌐 Web kullanım analizi
- 📦 Stok yönetimi

---
## 2. Temel Kavramlar

### 2.1 Support (Destek)
Bir ürünün ya da ürün kümesinin tüm işlemler içindeki görünme sıklığıdır.

$$Support(A) = \frac{A\ içeren\ işlem\ sayısı}{Toplam\ işlem\ sayısı}$$

### 2.2 Confidence (Güven)
A satın alındığında B'nin de satın alınma olasılığıdır.

$$Confidence(A \Rightarrow B) = \frac{Support(A \cup B)}{Support(A)}$$

### 2.3 Lift (Kaldıraç)
A ve B'nin birlikte görülmesinin rastlantısallıktan ne kadar uzak olduğunu gösterir.

$$Lift(A \Rightarrow B) = \frac{Confidence(A \Rightarrow B)}{Support(B)}$$

| Lift Değeri | Anlam |
|------------|-------|
| Lift > 1 | A ve B pozitif ilişkili ✅ |
| Lift = 1 | A ve B bağımsız ➡️ |
| Lift < 1 | A ve B negatif ilişkili ❌ |

In [ ]:
# Temel kavramları elle hesaplayalım

islemler = [
    ['ekmek', 'süt', 'tereyağ'],
    ['ekmek', 'süt'],
    ['süt', 'tereyağ', 'peynir'],
    ['ekmek', 'peynir'],
    ['ekmek', 'süt', 'peynir'],
    ['süt', 'tereyağ'],
    ['ekmek', 'süt', 'tereyağ', 'peynir'],
    ['ekmek', 'tereyağ'],
]

toplam = len(islemler)

def support(urun, veri):
    if isinstance(urun, list):
        sayi = sum(1 for i in veri if all(u in i for u in urun))
    else:
        sayi = sum(1 for i in veri if urun in i)
    return sayi / len(veri)

def confidence(A, B, veri):
    return support(A + B, veri) / support(A, veri)

def lift(A, B, veri):
    return confidence(A, B, veri) / support(B, veri)

A = ['ekmek']
B = ['süt']

print(f"Support(Ekmek)         : {support(A, islemler):.2f}")
print(f"Support(Süt)           : {support(B, islemler):.2f}")
print(f"Support(Ekmek ∪ Süt)   : {support(A+B, islemler):.2f}")
print(f"Confidence(Ekmek→Süt)  : {confidence(A, B, islemler):.2f}")
print(f"Lift(Ekmek→Süt)        : {lift(A, B, islemler):.2f}")

---
## 3. Apriori Algoritması

**Apriori**, birliktelik kuralı madenciliğinde kullanılan en klasik algoritmadır. 1994 yılında Agrawal ve Srikant tarafından geliştirilmiştir.

### Çalışma Prensibi
**"Apriori ilkesi"**: Eğer bir ürün kümesi sık geçmiyorsa, o kümenin hiçbir üst kümesi de sık geçmez.

### Adımlar
1. Minimum destek eşiğini belirle
2. Tek ürünlü sık kümeleri bul (L1)
3. Sık kümeleri birleştirerek aday kümeler oluştur (Ck+1)
4. Eşiği sağlamayanları budama
5. Sık kümeler kalmayana kadar tekrarla

### Dezavantajlar
- ⚠️ Büyük veri setlerinde yavaş çalışır
- ⚠️ Çok sayıda veritabanı taraması gerektirir
- ⚠️ Bellek kullanımı yüksek olabilir

In [ ]:
# Gerekli kütüphane kurulumu
# !pip install mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Örnek veri seti (market sepeti)
veri = [
    ['ekmek', 'süt', 'tereyağ'],
    ['ekmek', 'süt'],
    ['süt', 'tereyağ', 'peynir'],
    ['ekmek', 'peynir'],
    ['ekmek', 'süt', 'peynir'],
    ['süt', 'tereyağ'],
    ['ekmek', 'süt', 'tereyağ', 'peynir'],
    ['ekmek', 'tereyağ'],
    ['peynir', 'yumurta'],
    ['ekmek', 'yumurta', 'süt'],
]

# One-hot encoding
te = TransactionEncoder()
te_array = te.fit_transform(veri)
df = pd.DataFrame(te_array, columns=te.columns_)
print("One-Hot Encoded Veri:")
print(df)

In [ ]:
# Apriori ile sık kümeleri bul
sik_kumeler = apriori(df, min_support=0.3, use_colnames=True)
print("Sık Ürün Kümeleri (min_support=0.3):")
print(sik_kumeler.sort_values('support', ascending=False))

In [ ]:
# Birliktelik kurallarını çıkar
kurallar = association_rules(sik_kumeler, metric='lift', min_threshold=1.0)
kurallar = kurallar.sort_values('lift', ascending=False)

print("Birliktelik Kuralları:")
print(kurallar[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Support vs Confidence scatter plot
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    kurallar['support'],
    kurallar['confidence'],
    c=kurallar['lift'],
    cmap='YlOrRd',
    s=100,
    alpha=0.7,
    edgecolors='gray'
)
plt.colorbar(scatter, label='Lift')
plt.xlabel('Support', fontsize=12)
plt.ylabel('Confidence', fontsize=12)
plt.title('Apriori - Support vs Confidence (Renk: Lift)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('apriori_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafik kaydedildi: apriori_scatter.png")

---
## 4. ECLAT Algoritması

**ECLAT** (Equivalence Class Clustering and bottom-up Lattice Traversal), 2000 yılında Zaki tarafından önerilmiştir.

### Apriori'den Farkı
Apriori **yatay veri formatı** kullanırken, ECLAT **dikey veri formatı** (TID-listesi) kullanır.

| Format | Açıklama |
|--------|----------|
| Yatay | `İşlem → Ürünler` |
| Dikey | `Ürün → İşlem ID listesi (TID)` |

### Avantajları
- ✅ Veritabanını daha az tarar
- ✅ Küme işlemleriyle hızlı destek hesabı
- ✅ Apriori'den genellikle daha hızlı

### Dezavantajı
- ⚠️ TID listeleri çok fazla bellek tüketebilir

In [ ]:
# ECLAT - Dikey Format (TID Listesi) ile Elle Uygulama

def yatay_to_dikey(islemler):
    """Yatay formatı dikey (TID-list) formata çevirir"""
    tid_list = {}
    for tid, islem in enumerate(islemler):
        for urun in islem:
            if urun not in tid_list:
                tid_list[urun] = set()
            tid_list[urun].add(tid)
    return tid_list

def eclat(tid_list, min_support, toplam_islem):
    """Basit ECLAT implementasyonu"""
    sik_kumeler = {}
    
    # Tekli öğeleri filtrele
    tek_ogeler = {frozenset([k]): v for k, v in tid_list.items()
                  if len(v) / toplam_islem >= min_support}
    sik_kumeler.update(tek_ogeler)
    
    # Alt kümeden üst kümeye genişlet
    onceki = list(tek_ogeler.keys())
    
    while onceki:
        yeni = []
        for i in range(len(onceki)):
            for j in range(i + 1, len(onceki)):
                birlestir = onceki[i] | onceki[j]
                if len(birlestir) == len(onceki[i]) + 1:
                    tid_kesisim = tek_ogeler.get(onceki[i], tid_list.get(list(onceki[i])[0], set())) & \
                                  tek_ogeler.get(onceki[j], tid_list.get(list(onceki[j])[0], set()))
                    sup = len(tid_kesisim) / toplam_islem
                    if sup >= min_support and birlestir not in sik_kumeler:
                        sik_kumeler[birlestir] = tid_kesisim
                        yeni.append(birlestir)
        onceki = yeni
    
    return sik_kumeler

# TID listesini oluştur
tid_list = yatay_to_dikey(veri)

print("Dikey Format (TID Listesi):")
for urun, tidler in sorted(tid_list.items()):
    print(f"  {urun:12s} → İşlem ID'leri: {sorted(tidler)}")

In [ ]:
# ECLAT ile sık kümeleri bul
min_sup = 0.3
eclat_sonuc = eclat(tid_list, min_sup, len(veri))

print(f"\nECLAT Sık Ürün Kümeleri (min_support={min_sup}):")
print(f"{'Küme':<35} {'Destek':>10}")
print("-" * 47)
for kume, tidler in sorted(eclat_sonuc.items(), key=lambda x: -len(x[1])):
    sup = len(tidler) / len(veri)
    print(f"{str(set(kume)):<35} {sup:>10.2f}")

---
## 5. FP-Growth Algoritması

**FP-Growth** (Frequent Pattern Growth), Han ve arkadaşları tarafından 2000 yılında geliştirilmiştir.

### Temel Fikir
Veritabanını sadece **2 kez tarar** ve tüm veriyi **FP-Tree** adı verilen özel bir ağaç yapısına sıkıştırır. Böylece aday küme üretmeden sık kümeler bulunur.

### FP-Tree Yapısı
- Kök düğüm (null)
- Her dal bir işlemi temsil eder
- Ortak önekler paylaşılır (bellek tasarrufu)

### Avantajları
- ✅ Sadece 2 veritabanı taraması
- ✅ Aday küme üretmez → Hızlı
- ✅ Büyük veri setlerinde Apriori'den çok daha hızlı

### Dezavantajı
- ⚠️ FP-Tree bazen çok büyük olabilir
- ⚠️ Paralel işlemeye uyarlamak zordur

In [ ]:
# FP-Growth - mlxtend ile uygulama
from mlxtend.frequent_patterns import fpgrowth

fp_sik_kumeler = fpgrowth(df, min_support=0.3, use_colnames=True)
print("FP-Growth Sık Ürün Kümeleri (min_support=0.3):")
print(fp_sik_kumeler.sort_values('support', ascending=False).to_string(index=False))

In [ ]:
# FP-Growth ile birliktelik kuralları
fp_kurallar = association_rules(fp_sik_kumeler, metric='confidence', min_threshold=0.5)
fp_kurallar = fp_kurallar.sort_values('lift', ascending=False)

print("FP-Growth Birliktelik Kuralları:")
print(fp_kurallar[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))

In [ ]:
# FP-Tree yapısını görselleştir (basit)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

def dugum_ciz(ax, x, y, metin, renk='#4A90D9', boyut=14):
    circle = plt.Circle((x, y), 0.45, color=renk, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, metin, ha='center', va='center', fontsize=boyut,
            fontweight='bold', color='white', zorder=4)

def cizgi_ciz(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2+0.45), xytext=(x1, y1-0.45),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Kök
dugum_ciz(ax, 5, 7.2, 'null', renk='#2c3e50')

# Seviye 1
dugum_ciz(ax, 3, 5.5, 'ekmek:6', renk='#e74c3c')
dugum_ciz(ax, 7, 5.5, 'süt:2', renk='#27ae60')
cizgi_ciz(ax, 5, 7.2, 3, 5.5)
cizgi_ciz(ax, 5, 7.2, 7, 5.5)

# Seviye 2
dugum_ciz(ax, 1.5, 3.8, 'süt:4', renk='#27ae60')
dugum_ciz(ax, 4.5, 3.8, 'peynir:1', renk='#f39c12')
dugum_ciz(ax, 7, 3.8, 'trbğ:1', renk='#8e44ad')
cizgi_ciz(ax, 3, 5.5, 1.5, 3.8)
cizgi_ciz(ax, 3, 5.5, 4.5, 3.8)
cizgi_ciz(ax, 7, 5.5, 7, 3.8)

# Seviye 3
dugum_ciz(ax, 0.8, 2.1, 'trbğ:3', renk='#8e44ad')
dugum_ciz(ax, 2.2, 2.1, 'peynir:2', renk='#f39c12')
cizgi_ciz(ax, 1.5, 3.8, 0.8, 2.1)
cizgi_ciz(ax, 1.5, 3.8, 2.2, 2.1)

ax.set_title('FP-Tree Yapısı (Örnek)', fontsize=16, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('fp_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print("FP-Tree görseli kaydedildi.")

---
## 6. Algoritmaların Karşılaştırılması

| Özellik | Apriori | ECLAT | FP-Growth |
|---------|---------|-------|-----------|
| Veri Formatı | Yatay | Dikey (TID) | FP-Tree |
| DB Tarama Sayısı | Çok | Orta | 2 |
| Aday Küme Üretimi | Var | Var | Yok |
| Bellek Kullanımı | Orta | Yüksek | Orta |
| Hız (Büyük Veri) | Yavaş | Orta | Hızlı |
| Uygulama Kolaylığı | Kolay | Orta | Zor |
| Paralel İşleme | Uygun | Uygun | Zor |

In [ ]:
import time

# Apriori süresi
t1 = time.time()
sik_a = apriori(df, min_support=0.3, use_colnames=True)
apriori_sure = time.time() - t1

# FP-Growth süresi
t2 = time.time()
sik_fp = fpgrowth(df, min_support=0.3, use_colnames=True)
fp_sure = time.time() - t2

print("⏱️ Süre Karşılaştırması:")
print(f"  Apriori   : {apriori_sure*1000:.3f} ms  → {len(sik_a)} sık küme")
print(f"  FP-Growth : {fp_sure*1000:.3f} ms  → {len(sik_fp)} sık küme")
print(f"\nİki algoritma aynı {len(sik_a)} sık küme buldu mu? {len(sik_a) == len(sik_fp)}")

In [ ]:
# Karşılaştırma grafiği
import matplotlib.pyplot as plt
import numpy as np

kategoriler = ['DB Tarama\nSayısı', 'Bellek\nKullanımı', 'Hız\n(Büyük Veri)', 'Uygulama\nKolaylığı']
apriori_puanlar = [1, 3, 2, 5]   # 1=kötü, 5=iyi
eclat_puanlar   = [3, 2, 3, 3]
fp_puanlar      = [5, 3, 5, 2]

x = np.arange(len(kategoriler))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - width, apriori_puanlar, width, label='Apriori', color='#e74c3c', alpha=0.85)
b2 = ax.bar(x,         eclat_puanlar,   width, label='ECLAT',   color='#3498db', alpha=0.85)
b3 = ax.bar(x + width, fp_puanlar,      width, label='FP-Growth', color='#2ecc71', alpha=0.85)

ax.set_xlabel('Özellik', fontsize=12)
ax.set_ylabel('Puan (1=Kötü, 5=İyi)', fontsize=12)
ax.set_title('Birliktelik Kuralı Algoritmalarının Karşılaştırması', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(kategoriler, fontsize=11)
ax.set_ylim(0, 6)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in [b1, b2, b3]:
    for rect in bar:
        h = rect.get_height()
        ax.annotate(str(h), xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('algoritma_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafik kaydedildi.")

---
## 7. Gerçek Dünya Uygulaması: Market Sepeti Analizi

Daha büyük ve gerçekçi bir veri seti üzerinde tam bir analiz pipeline'ı uygulayalım.

In [ ]:
import pandas as pd
import numpy as np
import random
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

random.seed(42)

# Gerçekçi market ürünleri
urunler = [
    'ekmek', 'süt', 'yumurta', 'peynir', 'tereyağ', 'zeytin',
    'makarna', 'domates', 'biber', 'soğan', 'yoğurt', 'meyve suyu',
    'tavuk', 'kıyma', 'pirinç', 'şeker', 'un', 'ayçiçek yağı'
]

# 500 işlem üret
islemler = []
for _ in range(500):
    sepet_boyutu = random.randint(2, 7)
    sepet = random.sample(urunler, sepet_boyutu)
    # Gerçekçilik: ekmek+süt birlikte çok alınıyor
    if random.random() > 0.3:
        if 'ekmek' not in sepet: sepet.append('ekmek')
        if 'süt' not in sepet and random.random() > 0.4: sepet.append('süt')
    islemler.append(list(set(sepet)))

print(f"Toplam işlem sayısı: {len(islemler)}")
print(f"Örnek sepetler:")
for i in range(3):
    print(f"  Sepet {i+1}: {islemler[i]}")

In [ ]:
# Veriyi encode et
te = TransactionEncoder()
te_array = te.fit_transform(islemler)
df_market = pd.DataFrame(te_array, columns=te.columns_)

# FP-Growth ile analiz
sik = fpgrowth(df_market, min_support=0.15, use_colnames=True)
kurallar = association_rules(sik, metric='lift', min_threshold=1.2)
kurallar = kurallar.sort_values('lift', ascending=False)

print(f"Bulunan sık küme sayısı : {len(sik)}")
print(f"Bulunan kural sayısı    : {len(kurallar)}")
print("\n🏆 En Yüksek Lift'e Sahip İlk 10 Kural:")
print(kurallar[['antecedents','consequents','support','confidence','lift']]
      .head(10).to_string(index=False))

In [ ]:
# En sık görülen ürünleri görselleştir
urun_frekanslari = df_market.sum().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
renkler = plt.cm.Blues(np.linspace(0.4, 0.9, len(urun_frekanslari)))
bars = plt.bar(urun_frekanslari.index, urun_frekanslari.values,
               color=renkler[::-1], edgecolor='white', linewidth=0.5)
plt.xlabel('Ürün', fontsize=12)
plt.ylabel('Görünme Sayısı', fontsize=12)
plt.title('Market Sepetinde Ürün Frekansları', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('urun_frekanslari.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📝 Özet

| Algoritma | Ne Zaman Kullan? |
|-----------|------------------|
| **Apriori** | Küçük/orta veri setleri, öğrenme amaçlı |
| **ECLAT** | Dikey veri formatı varsa, orta büyüklükte veri |
| **FP-Growth** | Büyük veri setleri, üretim ortamı |

### Önemli Parametreler
- **min_support** → Çok düşük seçilirse kural sayısı patlar, çok yüksek seçilirse önemli kurallar kaçırılır
- **min_confidence** → Genellikle 0.5 ve üzeri anlamlı kabul edilir
- **min_lift** → 1'den büyük olması zorunludur, 1.2+ önerilir

---
*Hazırlayan: Buket Toptas | Veri Madenciliği Dersi*